In [1]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-chroma
!pip install -q chromadb
!pip install -q pypdf
!pip install -q langchain-groq

In [2]:
from google.colab import files

uploaded = files.upload()

Saving qpaper-sample.pdf to qpaper-sample (1).pdf


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("qpaper-sample.pdf")

documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 18


In [10]:
!pip install -q langchain-text-splitters

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 76


In [13]:
!pip install -q sentence-transformers

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_47522/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [18]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

tags=['Chroma', 'HuggingFaceEmbeddings'] vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7ebb7e612030> search_kwargs={'k': 3}


In [22]:
query = "Uses 5 carefully selected features"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n----- Chunk {i+1} -----\n")
    print(doc.page_content[:500])


----- Chunk 1 -----

added complexity of Model B (20 features vs 3 features) is justiﬁed by the performance
gains. [3 marks]

----- Chunk 2 -----

limitations. [4 marks]
(c) A company is building a perceptron-based classiﬁer to categorize customer reviews as
positive or negative. They have collected 500 training reviews and are comparing two
approaches:
Approach A: Uses 5 carefully selected features: count of positive sentiment words,
count of negative sentiment words, presence of exclamation marks, review length (word
count), and star rating (1-5).
Approach B: Uses 50 features representing the counts of the 50 most frequently

----- Chunk 3 -----

the model has learned features that are shared between birds and dogs but distinguish


In [27]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [30]:
# query = "What are the 5 carefully selected features used?"

# docs = retriever.invoke(query)

# context = "\n\n".join([doc.page_content for doc in docs])

# prompt = f"""
# Answer only from the provided context.

# Context:
# {context}

# Question:
# {query}

# Answer:
# """

# response = llm.invoke(prompt)

# print(response.content)
queries = ["What are the 5 carefully selected features used?", "Can you give me question 2", "What is Adam's apple?"]

for query in queries:
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""
    Answer only from the provided context.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    response = llm.invoke(prompt)
    print(response.content)
    print("================================================")

count of positive sentiment words, 
count of negative sentiment words, 
presence of exclamation marks, 
review length (word count), 
star rating (1-5).
2. (a) Consider a perceptron with 2 input features. Current weights are:
w =


w0
w1
w2

 =


0.5
0.3
−0.4



added complexity of Model B (20 features vs 3 features) is justiﬁed by the performance
gains. [3 marks]

23 dZ1 = dA1 * ( Z1 > 0)
24 dW1 = ____________ # Blank 5
25
26 return dW1 , dW2
There is no information about Adam's apple in the provided context.


In [26]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound-mini
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
meta-llama/llama-prompt-guard-2-22m
llama-3.3-70b-versatile
allam-2-7b
openai/gpt-oss-120b
whisper-large-v3-turbo
groq/compound
whisper-large-v3
llama-3.1-8b-instant
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-20b
